# FahMai RAG Advanced

In [ ]:
!unzip data.zip

## 1) Dependencies

In [ ]:
!pip install -U sentence-transformers pythainlp rank-bm25 requests python-dotenv
!pip install -U langchain-text-splitters


## 2) Imports


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
from google.colab import userdata

import numpy as np
import requests

try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

try:
    from sentence_transformers import SentenceTransformer
except Exception as exc:
    raise SystemExit(
        "Missing dependency: sentence-transformers\n"
        "Install with: pip install sentence-transformers"
    ) from exc

try:
    from pythainlp.tokenize import word_tokenize
except Exception as exc:
    raise SystemExit(
        "Missing dependency: pythainlp\n"
        "Install with: pip install pythainlp"
    ) from exc

try:
    from rank_bm25 import BM25Okapi
except Exception as exc:
    raise SystemExit(
        "Missing dependency: rank-bm25\n"
        "Install with: pip install rank-bm25"
    ) from exc

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    HAS_LANGCHAIN_SPLITTER = True
except Exception:
    HAS_LANGCHAIN_SPLITTER = False


## 3) Prompt + Config + Constants


In [ ]:
SYSTEM_PROMPT = """คุณคือระบบตอบคำถามหลายตัวเลือกของร้านฟ้าใหม่
กติกาบังคับ:
1) ใช้เฉพาะข้อมูลจากบริบทที่ให้มาเท่านั้น ห้ามเดา
2) เลือกคำตอบเพียงตัวเลือกเดียวจาก 1-10
3) ถ้าคำถามเกี่ยวกับร้านฟ้าใหม่ แต่ไม่พบข้อมูลเพียงพอให้สรุปคำตอบ -> ตอบ 9
4) ถ้าคำถามไม่เกี่ยวข้องกับร้านฟ้าใหม่/สินค้า/นโยบาย/ข้อมูลร้าน -> ตอบ 10
5) ออกผลเป็น JSON บรรทัดเดียวเท่านั้น รูปแบบ:
{"answer": <1-10>, "confidence": <0-1>, "evidence": ["C1","C3"], "reason": "..."}
"""

# =========================
# CONFIG (แก้ค่าที่นี่)
# =========================
DATA_DIR = "/content/data"
OUTPUT_CSV = "submission_advanced.csv"
DEBUG_JSONL = "debug_predictions.jsonl"
CACHE_DIR = ".cache_rag"

N_QUESTIONS = 100
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 500
TOP_K = 16
FETCH_K = 60

DENSE_MODEL = "intfloat/multilingual-e5-base"
# THAILLM_MODELS = ["typhoon", "openthaigpt"]  # เช่น ["typhoon"] เพื่อลดเวลา/ค่าใช้จ่าย
THAILLM_MODELS = ["openthaigpt"]
# THAILLM_API_KEY = ""  # ใส่ key ตรงนี้ หรือใช้ env THAILLM_API_KEY
THAILLM_API_KEY = userdata.get('ThaiLLM')
API_SLEEP_SEC = 0.25

RETRIEVAL_ONLY = False
RESUME_FROM_DEBUG = True


THAI_STOPWORDS = {
    "ครับ",
    "ค่ะ",
    "คะ",
    "นะ",
    "หน่อย",
    "หน่อยครับ",
    "หน่อยค่ะ",
    "ได้",
    "ไหม",
    "มั้ย",
    "หรือ",
    "แล้ว",
    "และ",
    "กับ",
    "ของ",
    "ที่",
    "ใน",
    "เป็น",
    "ให้",
    "จาก",
    "โดย",
    "แบบ",
    "มาก",
    "เลย",
    "อยาก",
    "ทราบ",
    "ขอ",
}

POLICY_HINT_WORDS = {
    "ประกัน",
    "เคลม",
    "คืนสินค้า",
    "ยกเลิก",
    "จัดส่ง",
    "สมาชิก",
    "คะแนน",
    "ชำระ",
    "ผ่อน",
    "ภาษี",
    "care",
    "on-site",
    "onsite",
    "บริการ",
    "trade-in",
    "เทิร์น",
}

STORE_HINT_WORDS = {
    "สาขา",
    "สำนักงานใหญ่",
    "ติดต่อ",
    "เบอร์",
    "โทร",
    "อีเมล",
    "line",
    "เวลาทำการ",
    "ก่อตั้ง",
    "พนักงาน",
    "รางวัล",
}

FAHMAI_ENTITY_HINTS = {
    "ฟ้าใหม่",
    "fahmai",
    "สายฟ้า",
    "saifah",
    "ดาวเหนือ",
    "daonuea",
    "คลื่นเสียง",
    "kluensiang",
    "วงโคจร",
    "wongkhojon",
    "จุดเชื่อม",
    "judchuam",
    "zenbyte",
    "novatech",
    "pulsegear",
    "arcwave",
}


## 4) Data Structures + Utility Functions


In [ ]:
@dataclass
class QuestionItem:
    qid: int
    question: str
    choices: Dict[str, str]


@dataclass
class Chunk:
    chunk_id: int
    text: str
    source: str
    category: str
    heading: str
    token_set: set
    source_terms: set


@dataclass
class LLMResult:
    model: str
    answer: Optional[int]
    confidence: float
    raw: str


def normalize_ws(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def normalize_for_match(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def simple_char_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    if len(text) <= chunk_size:
        return [text]
    pieces = []
    step = max(1, chunk_size - chunk_overlap)
    start = 0
    while start < len(text):
        piece = text[start : start + chunk_size]
        pieces.append(piece)
        start += step
    return pieces


def split_with_optional_langchain(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    if not text.strip():
        return []
    if HAS_LANGCHAIN_SPLITTER:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n### ", "\n## ", "\n\n", "\n", " ", ""],
        )
        return splitter.split_text(text)
    return simple_char_split(text, chunk_size=chunk_size, chunk_overlap=chunk_overlap)


def split_markdown_sections(text: str) -> Tuple[str, List[Tuple[str, str]]]:
    lines = text.splitlines()
    doc_title = ""
    current_heading = "ข้อมูลทั่วไป"
    buf: List[str] = []
    sections: List[Tuple[str, str]] = []

    def flush():
        nonlocal buf
        body = "\n".join(buf).strip()
        if body:
            sections.append((current_heading, body))
        buf = []

    for ln in lines:
        if ln.startswith("# "):
            doc_title = ln[2:].strip()
            continue
        if re.match(r"^#{2,6}\s+", ln):
            flush()
            current_heading = re.sub(r"^#{2,6}\s+", "", ln).strip()
            continue
        if ln.strip() == "---":
            continue
        buf.append(ln)
    flush()

    if not sections and text.strip():
        sections = [("ข้อมูลทั่วไป", text.strip())]
    return doc_title, sections


def tokenize_words(text: str) -> List[str]:
    tokens = word_tokenize(text, engine="newmm")
    out: List[str] = []
    for tk in tokens:
        tk = tk.strip().lower()
        if not tk:
            continue
        if tk in THAI_STOPWORDS:
            continue
        if re.fullmatch(r"[^\wก-๙]+", tk):
            continue
        out.append(tk)
    return out


def tokenize_chars(text: str, n: int = 3, max_tokens: int = 2000) -> List[str]:
    txt = re.sub(r"\s+", "", text.lower())
    txt = re.sub(r"[^a-z0-9ก-๙]", "", txt)
    if len(txt) < n:
        return [txt] if txt else []
    grams = [txt[i : i + n] for i in range(0, len(txt) - n + 1)]
    if len(grams) > max_tokens:
        step = max(1, len(grams) // max_tokens)
        grams = grams[::step][:max_tokens]
    return grams


def extract_source_terms(source_path: str, heading: str) -> set:
    stem = Path(source_path).stem.lower()
    parts = re.split(r"[_\-]+", stem)
    terms = set()
    for p in parts:
        p = p.strip()
        if not p:
            continue
        if len(p) >= 4 or any(ch.isdigit() for ch in p):
            terms.add(p)
    joined = "_".join(parts)
    bi = re.split(r"_+", joined)
    for i in range(len(bi) - 1):
        phrase = f"{bi[i]} {bi[i+1]}".strip()
        if len(phrase) >= 6:
            terms.add(phrase)
    for tk in tokenize_words(heading):
        if len(tk) >= 3:
            terms.add(tk)
    return terms


def load_questions(csv_path: Path, n_questions: int = 100) -> List[QuestionItem]:
    items: List[QuestionItem] = []
    with csv_path.open("r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            qid = int(row["id"])
            choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
            items.append(QuestionItem(qid=qid, question=row["question"], choices=choices))
    items = sorted(items, key=lambda x: x.qid)
    return items[:n_questions]



## 5) Chunk Builder


In [ ]:
def build_chunks(
    kb_dir: Path, chunk_size: int, chunk_overlap: int, min_chunk_chars: int = 80
) -> List[Chunk]:
    chunks: List[Chunk] = []
    cid = 0
    for fp in sorted(kb_dir.rglob("*.md")):
        raw = fp.read_text(encoding="utf-8")
        rel = str(fp.relative_to(kb_dir))
        category = rel.split("/", 1)[0] if "/" in rel else "unknown"
        doc_title, sections = split_markdown_sections(raw)

        for heading, section_text in sections:
            enriched = (
                f"[SOURCE] {rel}\n"
                f"[TITLE] {doc_title or Path(rel).stem}\n"
                f"[SECTION] {heading}\n\n"
                f"{section_text}"
            )
            pieces = split_with_optional_langchain(
                enriched, chunk_size=chunk_size, chunk_overlap=chunk_overlap
            )
            for piece in pieces:
                cleaned = piece.strip()
                if len(cleaned) < min_chunk_chars:
                    continue
                token_set = set(tokenize_words(cleaned))
                source_terms = extract_source_terms(rel, heading)
                chunks.append(
                    Chunk(
                        chunk_id=cid,
                        text=cleaned,
                        source=rel,
                        category=category,
                        heading=heading,
                        token_set=token_set,
                        source_terms=source_terms,
                    )
                )
                cid += 1
    return chunks


def maybe_prefix_e5(model_name: str, text: str, is_query: bool) -> str:
    lower = model_name.lower()
    if "e5" in lower:
        return f"{'query' if is_query else 'passage'}: {text}"
    return text


## 6) Hybrid Retriever


In [ ]:
class HybridRetriever:
    def __init__(
        self,
        chunks: Sequence[Chunk],
        dense_model_name: str,
        cache_dir: Path,
        word_weight: float = 1.0,
        char_weight: float = 0.9,
        dense_weight: float = 1.2,
        rrf_k: int = 60,
    ):
        self.chunks = list(chunks)
        self.dense_model_name = dense_model_name
        self.rrf_k = rrf_k
        self.word_weight = word_weight
        self.char_weight = char_weight
        self.dense_weight = dense_weight

        self.model = SentenceTransformer(dense_model_name)
        self.chunk_texts = [c.text for c in self.chunks]

        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.chunk_embeddings = self._load_or_build_embeddings()

        self.word_tokens = [tokenize_words(t) for t in self.chunk_texts]
        self.char_tokens = [tokenize_chars(t, n=3) for t in self.chunk_texts]
        self.bm25_word = BM25Okapi(self.word_tokens)
        self.bm25_char = BM25Okapi(self.char_tokens)

    def _embedding_cache_key(self) -> str:
        joined = f"{self.dense_model_name}|{len(self.chunk_texts)}|{sum(len(x) for x in self.chunk_texts)}"
        return hashlib.md5(joined.encode("utf-8")).hexdigest()

    def _load_or_build_embeddings(self) -> np.ndarray:
        cache_file = self.cache_dir / f"chunk_emb_{self._embedding_cache_key()}.npy"
        if cache_file.exists():
            return np.load(cache_file)

        docs = [maybe_prefix_e5(self.dense_model_name, t, is_query=False) for t in self.chunk_texts]
        embs = self.model.encode(
            docs,
            batch_size=32,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embs = np.asarray(embs, dtype=np.float32)
        np.save(cache_file, embs)
        return embs

    def _extract_choice_hints(self, choices: Dict[str, str]) -> str:
        merged = " ".join(choices[str(i)] for i in range(1, 9))
        toks = tokenize_words(merged)
        en = re.findall(r"[A-Za-z0-9\-\+\.]{2,}", merged)
        pool = [t for t in toks if len(t) >= 2] + [x.lower() for x in en]
        counts = Counter(pool)
        hints = [t for t, _ in counts.most_common(16)]
        return " ".join(hints)

    def _build_query(self, question: str, choices: Dict[str, str]) -> str:
        q = normalize_ws(question)
        hint = self._extract_choice_hints(choices)
        if hint:
            return f"{q}\nคำใบ้จากตัวเลือก: {hint}"
        return q

    @staticmethod
    def _top_idx(scores: np.ndarray, k: int) -> np.ndarray:
        if len(scores) <= k:
            return np.argsort(scores)[::-1]
        idx = np.argpartition(scores, -k)[-k:]
        idx = idx[np.argsort(scores[idx])[::-1]]
        return idx

    def retrieve(
        self,
        question: str,
        choices: Dict[str, str],
        top_k: int = 8,
        fetch_k: int = 36,
    ) -> Tuple[List[Chunk], Dict[str, float]]:
        q_full = self._build_query(question, choices)
        q_for_dense = maybe_prefix_e5(self.dense_model_name, q_full, is_query=True)
        q_emb = self.model.encode([q_for_dense], normalize_embeddings=True)
        dense_scores = np.dot(self.chunk_embeddings, q_emb.T).reshape(-1)
        dense_idx = self._top_idx(dense_scores, k=fetch_k)

        q_word = tokenize_words(q_full)
        q_char = tokenize_chars(q_full, n=3)
        word_scores = np.array(self.bm25_word.get_scores(q_word))
        char_scores = np.array(self.bm25_char.get_scores(q_char))
        word_idx = self._top_idx(word_scores, k=fetch_k)
        char_idx = self._top_idx(char_scores, k=fetch_k)

        rrf_scores = defaultdict(float)
        for rank, idx in enumerate(dense_idx, 1):
            rrf_scores[int(idx)] += self.dense_weight / (self.rrf_k + rank)
        for rank, idx in enumerate(word_idx, 1):
            rrf_scores[int(idx)] += self.word_weight / (self.rrf_k + rank)
        for rank, idx in enumerate(char_idx, 1):
            rrf_scores[int(idx)] += self.char_weight / (self.rrf_k + rank)

        query_norm = normalize_for_match(question)
        q_tokens = set(tokenize_words(question))
        choice_tokens = set(tokenize_words(" ".join(choices[str(i)] for i in range(1, 9))))

        is_policy_intent = any(w in query_norm for w in POLICY_HINT_WORDS)
        is_store_intent = any(w in query_norm for w in STORE_HINT_WORDS)

        all_candidates = set(map(int, dense_idx)) | set(map(int, word_idx)) | set(map(int, char_idx))
        for idx in all_candidates:
            ch = self.chunks[idx]

            # lexical overlap bonus
            q_overlap = len(q_tokens & ch.token_set) / (len(q_tokens) + 1e-6)
            c_overlap = len(choice_tokens & ch.token_set) / (len(choice_tokens) + 1e-6)
            rrf_scores[idx] += 0.10 * q_overlap + 0.04 * c_overlap

            # intent-category bonus
            if is_policy_intent and ch.category == "policies":
                rrf_scores[idx] += 0.03
            if is_store_intent and ch.category == "store_info":
                rrf_scores[idx] += 0.03

            # source-term match bonus
            term_hit = any(t in query_norm for t in ch.source_terms if len(t) >= 4)
            if term_hit:
                rrf_scores[idx] += 0.05

        fused_idx = sorted(rrf_scores.keys(), key=lambda i: rrf_scores[i], reverse=True)[:top_k]
        retrieved = [self.chunks[i] for i in fused_idx]

        stats = {
            "best_dense": float(dense_scores[dense_idx[0]]) if len(dense_idx) else -1.0,
            "best_word": float(word_scores[word_idx[0]]) if len(word_idx) else -1.0,
            "best_char": float(char_scores[char_idx[0]]) if len(char_idx) else -1.0,
        }
        return retrieved, stats


## 7) LLM + Parsing + Voting


In [ ]:
def build_prompt(q: QuestionItem, retrieved: Sequence[Chunk], max_context_chars: int = 900) -> str:
    context_blocks: List[str] = []
    for i, ch in enumerate(retrieved, 1):
        txt = ch.text.strip()
        if len(txt) > max_context_chars:
            txt = txt[:max_context_chars] + "\n... [ตัดทอน]"
        context_blocks.append(f"[C{i}] source={ch.source}\n{txt}")

    choices_text = "\n".join(f"{i}. {q.choices[str(i)]}" for i in range(1, 11))
    context = "\n\n".join(context_blocks)
    return (
        "บริบทจากฐานความรู้:\n"
        f"{context}\n\n"
        f"คำถาม:\n{q.question}\n\n"
        f"ตัวเลือก (1-10):\n{choices_text}\n\n"
        "ให้ตอบตามข้อมูลในบริบทเท่านั้น และคืนค่าเป็น JSON ตาม format ที่กำหนด"
    )


def ask_thaillm(
    api_key: str,
    model: str,
    messages: Sequence[Dict[str, str]],
    max_retries: int = 6,
    temperature: float = 0.0,
    timeout_sec: int = 120,
) -> str:
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": api_key}
    payload = {
        "model": "/model",
        "messages": list(messages),
        "max_tokens": 768,
        "temperature": temperature,
    }

    last_error = None
    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=timeout_sec)
            if resp.status_code == 429:
                wait = min(2**attempt, 30)
                time.sleep(wait)
                continue
            if 500 <= resp.status_code < 600:
                wait = min(2**attempt, 20)
                time.sleep(wait)
                continue
            resp.raise_for_status()
            data = resp.json()
            return data["choices"][0]["message"]["content"].strip()
        except Exception as exc:
            last_error = exc
            wait = min(2**attempt, 20)
            time.sleep(wait)
    raise RuntimeError(f"ThaiLLM API failed for model={model}: {last_error}")


def parse_llm_output(raw: str) -> Tuple[Optional[int], float]:
    text = raw.strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"^```json\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    answer = None
    confidence = 0.5

    for candidate in re.findall(r"\{[\s\S]*?\}", text):
        try:
            obj = json.loads(candidate)
        except Exception:
            continue
        if "answer" in obj:
            try:
                ans = int(obj["answer"])
                if 1 <= ans <= 10:
                    answer = ans
            except Exception:
                pass
        if "confidence" in obj:
            try:
                cf = float(obj["confidence"])
                confidence = max(0.0, min(1.0, cf))
            except Exception:
                pass
        if answer is not None:
            return answer, confidence

    m = re.search(r'"answer"\s*:\s*(\d+)', text)
    if m:
        ans = int(m.group(1))
        if 1 <= ans <= 10:
            answer = ans
            return answer, confidence

    m = re.search(r"ANSWER\s*:\s*(\d+)", text, flags=re.IGNORECASE)
    if m:
        ans = int(m.group(1))
        if 1 <= ans <= 10:
            answer = ans
            return answer, confidence

    for d in re.findall(r"\b(\d{1,2})\b", text):
        ans = int(d)
        if 1 <= ans <= 10:
            answer = ans
            break
    return answer, confidence


def aggregate_results(results: Sequence[LLMResult], primary_model: str) -> Optional[int]:
    votes = defaultdict(float)
    primary_answer = None
    for r in results:
        if r.answer is None:
            continue
        weight = max(0.05, min(1.0, r.confidence if r.confidence is not None else 0.5))
        votes[r.answer] += weight
        if r.model == primary_model:
            primary_answer = r.answer

    if not votes:
        return None
    best_score = max(votes.values())
    winners = [ans for ans, sc in votes.items() if abs(sc - best_score) <= 1e-9]
    if len(winners) == 1:
        return winners[0]
    if primary_answer in winners:
        return primary_answer
    return min(winners)


def fallback_answer(question: str, retrieval_stats: Dict[str, float]) -> int:
    qn = normalize_for_match(question)
    has_fahmai_term = any(k in qn for k in FAHMAI_ENTITY_HINTS)
    low_retrieval = (
        retrieval_stats.get("best_dense", -1.0) < 0.23
        and retrieval_stats.get("best_word", -1.0) < 4.0
        and retrieval_stats.get("best_char", -1.0) < 3.0
    )
    if low_retrieval and not has_fahmai_term:
        return 10
    return 9


def predict_one(
    q: QuestionItem,
    retrieved: Sequence[Chunk],
    api_key: str,
    models: Sequence[str],
    api_sleep_sec: float,
) -> Tuple[int, List[LLMResult]]:
    prompt = build_prompt(q, retrieved=retrieved)
    llm_results: List[LLMResult] = []
    primary_model = models[0]

    for idx, model in enumerate(models):
        local_prompt = prompt
        if idx % 2 == 1:
            # small perturbation in context order for robust voting
            rev_prompt = build_prompt(q, retrieved=list(retrieved)[::-1])
            local_prompt = rev_prompt

        raw = ask_thaillm(
            api_key=api_key,
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": local_prompt},
            ],
        )
        ans, conf = parse_llm_output(raw)
        llm_results.append(LLMResult(model=model, answer=ans, confidence=conf, raw=raw))
        if api_sleep_sec > 0:
            time.sleep(api_sleep_sec)

    pred = aggregate_results(llm_results, primary_model=primary_model)
    if pred is None:
        pred = 9
    return pred, llm_results


def write_submission(path: Path, predictions: Dict[int, int], total_questions: int = 100) -> None:
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "answer"])
        for qid in range(1, total_questions + 1):
            writer.writerow([qid, int(predictions.get(qid, 9))])


def append_debug(debug_path: Path, payload: Dict) -> None:
    with debug_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")



## 8) Pipeline Runner


In [ ]:
def main() -> None:
    if load_dotenv is not None:
        load_dotenv()

    data_dir = Path(DATA_DIR).resolve()
    kb_dir = data_dir / "knowledge_base"
    questions_path = data_dir / "questions.csv"
    output_path = Path(OUTPUT_CSV).resolve()
    debug_path = Path(DEBUG_JSONL).resolve()
    cache_dir = Path(CACHE_DIR).resolve()

    if not kb_dir.exists():
        raise SystemExit(f"KB directory not found: {kb_dir}")
    if not questions_path.exists():
        raise SystemExit(f"questions.csv not found: {questions_path}")

    api_key = THAILLM_API_KEY.strip() or os.environ.get("THAILLM_API_KEY", "").strip()
    if not RETRIEVAL_ONLY and not api_key:
        raise SystemExit("Missing ThaiLLM API key. Set THAILLM_API_KEY in CONFIG or env THAILLM_API_KEY")

    models = [m.strip() for m in THAILLM_MODELS if str(m).strip()]
    if not models:
        raise SystemExit("No model provided in THAILLM_MODELS")

    print("Loading questions...")
    questions = load_questions(questions_path, n_questions=N_QUESTIONS)
    print(f"Loaded {len(questions)} questions")

    print("Building chunks...")
    chunks = build_chunks(
        kb_dir=kb_dir,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    print(f"Built {len(chunks)} chunks")

    print("Building retriever indexes...")
    retriever = HybridRetriever(
        chunks=chunks,
        dense_model_name=DENSE_MODEL,
        cache_dir=cache_dir,
    )

    predictions: Dict[int, int] = {}
    if RESUME_FROM_DEBUG and debug_path.exists():
        with debug_path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except Exception:
                    continue
                if "id" in obj and "pred" in obj:
                    predictions[int(obj["id"])] = int(obj["pred"])
        if predictions:
            print(f"Resumed {len(predictions)} answered questions from debug log")

    total = len(questions)
    for n, q in enumerate(questions, 1):
        if q.qid in predictions:
            print(f"[{n:03d}/{total:03d}] Q{q.qid}: skip (resumed)")
            continue

        retrieved, stats = retriever.retrieve(
            question=q.question,
            choices=q.choices,
            top_k=TOP_K,
            fetch_k=FETCH_K,
        )

        if RETRIEVAL_ONLY:
            pred = fallback_answer(q.question, stats)
            llm_results: List[LLMResult] = []
        else:
            try:
                pred, llm_results = predict_one(
                    q=q,
                    retrieved=retrieved,
                    api_key=api_key,
                    models=models,
                    api_sleep_sec=API_SLEEP_SEC,
                )
            except Exception as exc:
                print(f"  LLM failed on Q{q.qid}: {exc}")
                pred = fallback_answer(q.question, stats)
                llm_results = []

        if pred not in range(1, 11):
            pred = fallback_answer(q.question, stats)

        predictions[q.qid] = pred
        write_submission(output_path, predictions, total_questions=100)

        dbg = {
            "id": q.qid,
            "pred": pred,
            "question": q.question,
            "stats": stats,
            "top_sources": [ch.source for ch in retrieved[:5]],
            "llm": [
                {"model": r.model, "answer": r.answer, "confidence": r.confidence, "raw": r.raw}
                for r in llm_results
            ],
        }
        append_debug(debug_path, dbg)
        print(f"[{n:03d}/{total:03d}] Q{q.qid}: pred={pred}")

    write_submission(output_path, predictions, total_questions=100)
    print(f"Done. Submission written to: {output_path}")
    print(f"Debug log written to: {debug_path}")


## 9) Run


In [ ]:
main()


Loading questions...
Loaded 100 questions
Building chunks...
Built 1005 chunks
Building retriever indexes...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[001/100] Q1: pred=5
[002/100] Q2: pred=7
[003/100] Q3: pred=2
[004/100] Q4: pred=6
[005/100] Q5: pred=6
[006/100] Q6: pred=8
[007/100] Q7: pred=1
[008/100] Q8: pred=4
[009/100] Q9: pred=4
[010/100] Q10: pred=2
[011/100] Q11: pred=4
[012/100] Q12: pred=1
[013/100] Q13: pred=1
[014/100] Q14: pred=4
[015/100] Q15: pred=7
[016/100] Q16: pred=1
[017/100] Q17: pred=8
[018/100] Q18: pred=5
[019/100] Q19: pred=2
[020/100] Q20: pred=2
[021/100] Q21: pred=3
[022/100] Q22: pred=6
[023/100] Q23: pred=3
[024/100] Q24: pred=6
[025/100] Q25: pred=5
[026/100] Q26: pred=6
[027/100] Q27: pred=2
[028/100] Q28: pred=7
[029/100] Q29: pred=4
[030/100] Q30: pred=3
[031/100] Q31: pred=4
[032/100] Q32: pred=2
[033/100] Q33: pred=8
[034/100] Q34: pred=5
[035/100] Q35: pred=3
[036/100] Q36: pred=2
[037/100] Q37: pred=9
[038/100] Q38: pred=6
[039/100] Q39: pred=4
[040/100] Q40: pred=8
[041/100] Q41: pred=7
[042/100] Q42: pred=2
[043/100] Q43: pred=4
[044/100] Q44: pred=1
[045/100] Q45: pred=3
[046/100] Q46: pred